# Sentiment Analysis: From Social Media to Student Communications
## Assessing Performance on Real-World Gmail and WhatsApp Data

**Project Team: Group K**
**Date:** May 2026

---

## 1. Problem Definition & Introduction
### The Challenge
Generic sentiment models often struggle with student data. This project develops a system to process academic contexts—such as rejections, deadlines, and alerts—across formal (Gmail) and informal (WhatsApp) channels.

## Objectives of the notebook
1. Adapt models to the academic domain by fine-tuning on student-specific vocabulary.
2. Benchmark classical algorithms against Transformers to compare their performance.
3. Use SHAP and LIME to explain model decisions and ensure interpretability.

## 2. Importing the necessary Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="pastel")



KeyboardInterrupt: 

## 3. Data Loading

In [ ]:
# Training data
df_sentiment = pd.read_csv('data/raw/sentimentdataset.csv')
df_tweets = pd.read_csv('data/raw/Tweets.csv')
df_twitterdataset = pd.read_csv('data/raw/twitterdataset.csv', header=None)

df_twitterdataset.columns  = ["id",'topic',"sentiment","text"]


# Gmail and whatsapp data for testing the models
df_gmail = pd.read_csv("data/raw/gmail_raw.csv")
df_whatsapp = pd.read_csv("/data/raw/whatsapp_raw.csv")


## 4. Exploratory Data Analysis (EDA)

## 4.1 EDA for Training Dataset

### Shape of the data

In [ ]:
print("=== RAW TRAINIGN AND VALIDATION DATA LOADED ===")
print(f"sentimentdataset.csv  → {df_sentiment.shape[0]} rows, {df_sentiment.shape[1]} columns")
print(f"Tweets.csv            → {df_tweets.shape[0]} rows, {df_tweets.shape[1]} columns")
print(f"twitterdataset.csv    → {df_twitterdataset.shape[0]} rows, {df_twitterdataset.shape[1]} columns")


### Columns available for the three datasets

In [ ]:

print("sentimentdataset.csv columns:", df_sentiment.columns.tolist())
print("Tweets.csv columns:", df_tweets.columns.tolist())
print("twitterdataset.csv columns:", df_twitterdataset.columns.tolist())


print("gmail_raw.csv columns:", df_gmail.columns.to_list())
print("whatsapp_raw.csv:", df_whatsapp.columns.to_list())
 

### checking for null values

In [ ]:
print("\n=== NULL VALUES CHECK ===")
print("df_sentiment nulls:\n", df_sentiment.isnull().sum())
print()
print("df_tweets nulls:\n", df_tweets.isnull().sum())
print()
print("df_twitterdataset nulls:\n", df_twitterdataset.isnull().sum())
print()


### Checking for rows where the text is just whitespace 



In [ ]:
df_sentiment = df_sentiment[df_sentiment['text'].str.strip().str.len() > 0]
df_tweets = df_tweets[df_tweets['text'].str.strip().str.len() > 0]
df_twitterdataset = df_twitterdataset[df_twitterdataset['text'].str.strip().str.len() > 0]

### Checking the class distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Keep the same class order and colors across all plots
sentiment_order = ['Positive', 'Neutral', 'Negative']
sentiment_colors = {
    'Positive': '#4CAF50',
    'Neutral': '#2196F3',
    'Negative': '#F44336'
}
bar_colors = [sentiment_colors[s] for s in sentiment_order]

# Dataset 1: sentimentdataset.csv
df_sentiment['sentiment'].value_counts().reindex(sentiment_order, fill_value=0).plot(
    kind='bar', ax=axes[0], color=bar_colors, edgecolor='black', width=0.6
)
axes[0].set_title('sentimentdataset.csv — Label Distribution', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Sentiment Class')
axes[0].set_ylabel('Number of Rows')
axes[0].tick_params(axis='x', rotation=0)
for p in axes[0].patches:
    axes[0].annotate(
        str(int(p.get_height())),
        (p.get_x() + p.get_width() / 2.0, p.get_height()),
        ha='center', va='bottom', fontsize=10
    )

# Dataset 2: Tweets.csv
df_tweets['sentiment'].value_counts().reindex(sentiment_order, fill_value=0).plot(
    kind='bar', ax=axes[1], color=bar_colors, edgecolor='black', width=0.6
)
axes[1].set_title('Tweets.csv — Label Distribution', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Sentiment Class')
axes[1].set_ylabel('Number of Rows')
axes[1].tick_params(axis='x', rotation=0)
for p in axes[1].patches:
    axes[1].annotate(
        str(int(p.get_height())),
        (p.get_x() + p.get_width() / 2.0, p.get_height()),
        ha='center', va='bottom', fontsize=10
    )

# Dataset 3: twitterdataset.csv
df_twitterdataset['sentiment'].value_counts().reindex(sentiment_order, fill_value=0).plot(
    kind='bar', ax=axes[2], color=bar_colors, edgecolor='black', width=0.6
)
axes[2].set_title('twitterdataset.csv — Label Distribution', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Sentiment Class')
axes[2].set_ylabel('Number of Rows')
axes[2].tick_params(axis='x', rotation=0)
for p in axes[2].patches:
    axes[2].annotate(
        str(int(p.get_height())),
        (p.get_x() + p.get_width() / 2.0, p.get_height()),
        ha='center', va='bottom', fontsize=10
    )

plt.suptitle('Class Distribution in Each Dataset Before Merging', fontsize=14, y=1.03)
plt.tight_layout()
plt.savefig('viz1_distribution_before_merge.png', dpi=150, bbox_inches='tight')
plt.show()

## 4.2 EDA for the Test Data

# implemet theothe eda xtics for the test data

In [ ]:
print("=== RAW TEST DATA LOADED ===")
print(f"Raw Gmail records: {len(df_gmail)}")
print(f"Raw WhatsApp records: {len(df_whatsapp)}")

## 5. Data Cleaning And Preprocessing

In [ ]:
# Maps any sentiment label to one of three standard classes: Positive, Negative, or Neutral and Returns None for labels we cannot confidently map.

def normalize_label(label):
    
    label = str(label).strip().lower()
 
    positive_labels = [
        'positive', 'joy', 'happiness', 'happy', 'love', 'excitement',
        'excited', 'contentment', 'content', 'gratitude', 'grateful',
        'serenity', 'serene', 'hopeful', 'hope', 'awe', 'pride',
        'acceptance', 'relief', 'mischievous', 'enthusiasm'
    ]
 
    negative_labels = [
        'negative', 'anger', 'angry', 'fear', 'sadness', 'sad',
        'disgust', 'disgusted', 'grief', 'despair', 'loneliness',
        'lonely', 'embarrassed', 'embarrassment', 'hate', 'bad',
        'nostalgia', 'confusion', 'confused', 'anxiety', 'anxious'
    ]
 
    neutral_labels = [
        'neutral', 'curiosity', 'curious', 'surprise', 'surprised'
    ]
 
    if label in positive_labels:
        return 'Positive'
    elif label in negative_labels:
        return 'Negative'
    elif label in neutral_labels:
        return 'Neutral'
    else:
        # Any label we cannot confidently classify is dropped
        return None
 

## 5.1 Training Data Preprocessing and Cleaning

### Dropping useless Columns for the trainiing datasets

In [ ]:
df_sentiment = df_sentiment[['Text', 'Sentiment']]
df_tweets = df_tweets[['text', 'airline_sentiment']]
df_twitterdataset = df_twitterdataset[['text', 'sentiment']]

### Renaming the training dataset columns to a consistent standard 

In [ ]:
df_sentiment = df_sentiment.rename(columns={'Text': 'text', 'Sentiment': 'sentiment'})
df_tweets = df_tweets.rename(columns={'airline_sentiment': 'sentiment'})
# twitter dataset columns are already intact

### Viewing the available columns

In [ ]:
print("\n=== AFTER DROPPING USELESS COLUMNS ===")
print(f"df_sentiment columns: {df_sentiment.columns.tolist()}")
print(f"df_tweets columns: {df_tweets.columns.tolist()}")
print(f"df_twitterdataset columns: {df_twitterdataset.columns.tolist()}")


### Dropping rows where text or sentiment is null

In [ ]:
df_social = df_sentiment.dropna(subset=['text', 'sentiment'])
df_tweets = df_tweets.dropna(subset=['text', 'sentiment'])
df_twitterdataset = df_twitterdataset.dropna(subset=['text', 'sentiment'])


### Remove Duplicates with in each dataset

In [ ]:

print("\n=== REMOVING DUPLICATES ===")
for name, df in [("df_sentiment", df_sentiment), ("df_tweets", df_tweets), ("df_twitterdataset", df_twitterdataset)]:
    before = len(df)
    df.drop_duplicates(subset=['text'], inplace=True)
    print(f"{name}: {before} -> {len(df)} records")


print("\n=== DUPLICATES CHECK (BEFORE REMOVAL) ===")
print(f"df_sentiment duplicates: {df_sentiment.duplicated(subset='text').sum()}")
print(f"df_tweets duplicates: {df_tweets.duplicated(subset='text').sum()}")
print(f"df_twitterdataset duplicates: {df_twitterdataset.duplicated(subset='text').sum()}")



df_sentiment = df_sentiment.drop_duplicates(subset='text')
df_tweets = df_tweets.drop_duplicates(subset='text')
df_twitterdataset = df_twitterdataset.drop_duplicates(subset='text')


print("\nAfter removing duplicates within each dataset:")
print(f"df_sentiment → {df_sentiment.shape[0]} rows")
print(f"df_tweets → {df_tweets.shape[0]} rows")
print(f"df_twitterdataset → {df_twitterdataset.shape[0]} rows")

### Applying the normalise label function to map the classes 

In [ ]:
df_sentiment['sentiment'] = df_sentiment['sentiment'].apply(normalize_label)
df_tweets['sentiment'] = df_tweets['sentiment'].apply(normalize_label)
df_twitterdataset['sentiment'] = df_twitterdataset['sentiment'].apply(normalize_label)


### Dropping rows where label could not be mapped and also viewing what was changed and what was not 

In [ ]:
before_sentiment = len(df_sentiment)
before_tweets = len(df_tweets)
before_twitterdataset = len(df_twitterdataset)

 
df_sentiment = df_sentiment.dropna(subset=['sentiment'])
df_tweets = df_tweets.dropna(subset=['sentiment'])
df_twitterdataset = df_twitterdataset.dropna(subset=['sentiment'])
 
print("\n=== LABEL NORMALISATION ===")
print(f"df_sentiment: {before_sentiment} → {len(df_sentiment)} rows (dropped {before_sentiment - len(df_sentiment)} unmappable labels)")
print(f"df_tweets: {before_tweets} → {len(df_tweets)} rows (dropped {before_tweets - len(df_tweets)} unmappable labels)")
print(f"df_twitterdataset: {before_twitterdataset} → {len(df_twitterdataset)} rows (dropped {before_twitterdataset - len(df_twitterdataset)} unmappable labels)")

print("\ndf_sentiment label distribution after normalisation:")
print(df_sentiment['sentiment'].value_counts())
 
print("\ndf_tweets label distribution after normalisation:")
print(df_tweets['sentiment'].value_counts())

print("\ndf_twitterdataset label distribution after normalisation:")
print(df_twitterdataset['sentiment'].value_counts())
 
 

### Merging the Datasets into one 

In [ ]:
# MERGE THE TWO DATASETS

df_sentiment['source'] = 'social_media'
df_tweets['source'] = 'airline_tweets'
df_twitterdataset['source'] = 'twitterdataset'

combined_df = pd.concat([df_sentiment, df_tweets, df_twitterdataset], ignore_index=True)

print("\n=== AFTER MERGING ===")
print(f"Combined dataset → {combined_df.shape[0]} rows")
print("\nSource breakdown:")
print(combined_df['source'].value_counts())

### Removing cross dataset duplicates

In [ ]:
before_merge_dedup = len(combined_df)
combined_df = combined_df.drop_duplicates(subset='text').reset_index(drop=True)
print(f"\nCross-dataset duplicates removed: {before_merge_dedup - len(combined_df)}")
print(f"Combined dataset after removing duplicates → {combined_df.shape[0]} rows")

# Save a copy BEFORE balancing for later comparison charts
combined_before_balance = combined_df.copy()


### Balancing the Dataset by Reducing Negative Samples

In [ ]:

# Reduce Negative class by 5000 samples in the combined dataset
negative_mask = combined_df['sentiment'] == 'Negative'
negative_count_before = int(negative_mask.sum())
remove_n = min(5000, negative_count_before)

if remove_n > 0:
    drop_idx = combined_df[negative_mask].sample(n=remove_n, random_state=42).index
    combined_df = combined_df.drop(index=drop_idx).reset_index(drop=True)

# Save a copy AFTER balancing
combined_balanced = combined_df.copy()

negative_count_after = int((combined_df['sentiment'] == 'Negative').sum())
print(f"Negative samples removed: {remove_n}")
print(f"Negative count: {negative_count_before} → {negative_count_after}")
print(f"Combined dataset after downsampling → {combined_df.shape[0]} rows")

### Class Imbalance check

In [ ]:
print("\n=== CLASS IMBALANCE CHECK ===")
class_counts = combined_df['sentiment'].value_counts()
print(class_counts)
 
majority = class_counts.max()
minority = class_counts.min()
imbalance_ratio = majority / minority
print(f"\nImbalance ratio (majority / minority): {imbalance_ratio:.2f}x")
 
if imbalance_ratio > 2:
    print("  Imbalance detected — balancing required")
else:
    print(" Classes are reasonably balanced")

### Class Distribution after merging

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
 
# Bar chart
colors = ['#4CAF50', '#F44336', '#2196F3']
class_counts.plot(kind='bar', ax=axes[0], color=colors, edgecolor='black', width=0.55)
axes[0].set_title('Merged Dataset — Class Distribution (Bar)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Sentiment Class')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)
for p in axes[0].patches:
    axes[0].annotate(str(int(p.get_height())),
                     (p.get_x() + p.get_width() / 2., p.get_height()),
                     ha='center', va='bottom', fontsize=11)
 
# Pie chart
axes[1].pie(
    class_counts.values,
    labels=class_counts.index,
    autopct='%1.1f%%',
    colors=colors,
    startangle=140,
    wedgeprops={'edgecolor': 'white', 'linewidth': 1.5}
)
axes[1].set_title('Merged Dataset — Class Distribution (Pie)', fontsize=13, fontweight='bold')
 
plt.suptitle('Class Distribution After Merging', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('viz2_distribution_after_merge.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: viz2_distribution_after_merge.png")
 
 

###   Before vs After balancing visualisation (side by side)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sentiment_order = ['Positive', 'Neutral', 'Negative']
sentiment_colors = {
    'Positive': '#4CAF50',
    'Neutral': '#2196F3',
    'Negative': '#F44336'
}
bar_colors = [sentiment_colors[s] for s in sentiment_order]

before_counts = combined_before_balance['sentiment'].value_counts().reindex(sentiment_order, fill_value=0)
after_counts = combined_balanced['sentiment'].value_counts().reindex(sentiment_order, fill_value=0)

before_counts.plot(kind='bar', ax=axes[0], color=bar_colors, edgecolor='black', width=0.55)
axes[0].set_title('Before Balancing', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Sentiment')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)
for p in axes[0].patches:
    axes[0].annotate(
        str(int(p.get_height())),
        (p.get_x() + p.get_width() / 2.0, p.get_height()),
        ha='center', va='bottom', fontsize=11
    )

after_counts.plot(kind='bar', ax=axes[1], color=bar_colors, edgecolor='black', width=0.55)
axes[1].set_title('After Balancing', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Sentiment')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=0)
for p in axes[1].patches:
    axes[1].annotate(
        str(int(p.get_height())),
        (p.get_x() + p.get_width() / 2.0, p.get_height()),
        ha='center', va='bottom', fontsize=11
    )

plt.suptitle('Class Balancing Effect', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('viz3_balancing_effect.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: viz3_balancing_effect.png')

###  Text length distribution by sentiment class

In [ ]:
# This helps us understand whether positive/negative/neutral posts
# tend to be longer or shorter — useful context for the report.
combined_balanced['text_length'] = combined_balanced['text'].str.split().str.len()
 
plt.figure(figsize=(10, 5))
for label, color in zip(['Positive', 'Negative', 'Neutral'], colors):
    subset = combined_balanced[combined_balanced['sentiment'] == label]['text_length']
    sns.kdeplot(subset, label=label, color=color, fill=True, alpha=0.3)
 
plt.title('Text Length Distribution by Sentiment Class', fontsize=13, fontweight='bold')
plt.xlabel('Number of Words')
plt.ylabel('Density')
plt.legend(title='Sentiment')
plt.tight_layout()
plt.savefig('viz4_text_length_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: viz4_text_length_distribution.png")

### Source contribution to final dataset visualisation



In [ ]:
# Shows how much each original dataset contributed after balancing.
 
source_sentiment = combined_balanced.groupby(
    ['source', 'sentiment']
).size().unstack(fill_value=0)
 
source_sentiment.plot(
    kind='bar', color=colors, edgecolor='black', width=0.6, figsize=(9, 5)
)
plt.title('Source Contribution per Sentiment Class', fontsize=13, fontweight='bold')
plt.xlabel('Data Source')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.legend(title='Sentiment')
plt.tight_layout()
plt.savefig('viz5_source_contribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: viz5_source_contribution.png")

### Split and Save Training and Validation sets

In [ ]:
final_df = combined_balanced[['text', 'sentiment']].copy()

train_df, val_df = train_test_split(
    final_df,
    test_size=0.30,
    random_state=42,
    stratify=final_df['sentiment']
)

train_path = 'data/processed/processed_training_dataset.csv'
val_path = 'data/processed/processed_validation_datset.csv'

train_df.to_csv(train_path, index=False)
val_df.to_csv(val_path, index=False)

print("\n=== FINAL SUMMARY ===")
print(f"Training dataset saved: {train_path} — {len(train_df)} rows")
print(f"Validation dataset saved: {val_path} — {len(val_df)} rows")

print("\nTraining class distribution:")
print(train_df['sentiment'].value_counts())

print("\nValidation class distribution:")
print(val_df['sentiment'].value_counts())

## Test Data Preprocessing and Cleaning

# to do is impement the 

### Dropping Duplicates

In [ ]:

df_gmail = df_gmail.drop_duplicates(subset=['text']).reset_index(drop=True)
df_whatsapp = df_whatsapp.drop_duplicates(subset=['text']).reset_index(drop=True)

print(f"Gmail records after deduplication: {len(df_gmail)}")
print(f"WhatsApp records after deduplication: {len(df_whatsapp)}")

 We define a robust cleaning function using regular expressions to strip signatures, HTML, and specific noise markers identified during EDA.

In [ ]:
def clean_the_text(text):
    # Remove HTML tags (<div>, <br>, etc.)
    text = re.sub(r'<.*?>', '', text)
    
    # Remove Forwarded message markers and signatures
    text = re.sub(r'--- Forwarded message ---', '', text)
    text = re.sub(r'Please consider the environment', '', text, flags=re.I)
    text = re.sub(r'Sent from my (iPhone|mobile|Android|iPhone)', '', text, flags=re.I)
    text = re.sub(r'Best regards, .*', '', text, flags=re.I)
    
    # Remove service markers (MTN:, GitHub:, etc.)
    text = re.sub(r'(MTN|GitHub|Vercel|Udemy|Vultr|CodeMagic|LinkedIn|Amazon|Railway|Netlify|Heroku):', '', text, flags=re.I)

    # Remove specific noise characters (!!!, ???, @user)
    text = re.sub(r'!!!|\?\?\?|@user', '', text)
    
    # Remove excessive whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text


### Applying the clean the text fucntion to the datasets

In [ ]:
df_gmail['clean_text'] = df_gmail['text'].apply(clean_the_text)
df_whatsapp['clean_text'] = df_whatsapp['text'].apply(clean_the_text)


### First five columns of the Data

In [ ]:

df_gmail[['text', 'clean_text']].head()

## Dropping the columns that are not needed & Merging the two datasets
We drop the 'noise' columns (meta_data, priority, is_starred, etc.) and merge the two streams into a unified testing dataset.

In [ ]:
# Tag sources before merging for visualization
df_gmail['source'] = 'Gmail'
df_whatsapp['source'] = 'WhatsApp'

# Select final columns
gmail_final = df_gmail[['clean_text', 'sentiment', 'source']].rename(columns={'clean_text': 'text'})
whatsapp_final = df_whatsapp[['clean_text', 'sentiment', 'source']].rename(columns={'clean_text': 'text'})

# Merge
test_dataset = pd.concat([gmail_final, whatsapp_final], ignore_index=True)
test_dataset = test_dataset.drop_duplicates(subset=['text']).reset_index(drop=True)
print(f"Final Test Dataset Size after deduplication: {len(test_dataset)}")

### Visualising the test Data distribution

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16, 6))

# 1. Source Contribution
sns.countplot(data=test_dataset, x='source', ax=ax[0])
ax[0].set_title('Test Data Source Distribution')
ax[0].set_ylabel('Sample Count')

# 2. Sentiment Distribution (Verification of Balance)
sns.countplot(data=test_dataset, x='sentiment', ax=ax[1], order=['Positive', 'Neutral', 'Negative'])
ax[1].set_title('Sentiment Distribution in Real-World Test Set from gmail and whatsapp')
ax[1].set_ylabel('Sample Count')

plt.show()

### Test length Analysis

In [ ]:
test_dataset['char_count'] = test_dataset['text'].apply(len)

plt.figure(figsize=(10, 6))
sns.kdeplot(data=test_dataset, x='char_count', hue='source', fill=True)
plt.title('Text Length Distribution: Gmail vs. WhatsApp')
plt.xlabel('Character Count')
plt.show()

In [ ]:
output_path = '../data/processed/student_test_dataset.csv'
test_dataset[['text', 'sentiment']].to_csv(output_path, index=False)
print(f"Success! Cleaned test set saved to: {output_path}")

### Exporting to the processed data folder

## 6. Feature Engineering Strategy
### 6.1 Classical Vectorization (TF-IDF/N-Grams)
*Preprocessing for Naive Bayes, LogReg, SVM, and Random Forest.*

### 6.2 Sequence Tokenization (Embeddings)
*Preprocessing for LSTM, GRU, TextCNN, and BERT.*

## 7. Model Training & Evaluation (Ordered by Complexity)
---

### Stage 7.1: Multinomial Naive Bayes (Larry)
*Baseline probabilistic classification.*

### Stage 7.2: Logistic Regression (Ritah)
*Linear classification with L1/L2 regularization.*

### Stage 7.3: Support Vector Machine - SVM (Ivy)
*High-dimensional separation using RBF kernels.*

### Stage 7.4: Random Forest (Julianah)
*Ensemble decision tree learning.*

### Stage 7.5: TextCNN (Larry)
*Deep Learning: 1D Convolutional Neural Network for text.*

### Stage 7.6: Gated Recurrent Unit - GRU (Julianah)
*Deep Learning: Efficient recurrent modeling.*

### Stage 7.7: Long Short-Term Memory - LSTM (Ritah)
*Deep Learning: Sequence modeling with memory gates.*

### Stage 7.8: Bidirectional LSTM - Bi-LSTM (Ivy)
*Deep Learning: Dual-direction context modeling.*

### Stage 7.9: DistilBERT with Domain Adaptation (David)
*Transformer-based modeling: Masked Language Modeling and Fine-Tuning.*

## 8. Comparative Analysis & Performance Comparison
*One comprehensive table/chart comparing all 9 models on Accuracy and F1-Score.*

## 9. Model Interpretability (SHAP & LIME)
*Explaining word-level attribution for the best-performing model.*

## 10. Domain Stress-Test
*Inference on real-world unseen student narratives.*

## 11. Conclusion & Technical Recommendations
*Summary of which model is best for production and why.*